# Defining Features (X) and Target (y) Correctly

Before training any supervised model, answer two questions clearly:
1. What exactly am I predicting?
2. What information is legitimately available at prediction time?

This notebook turns those questions into an actionable workflow to prevent leakage and build production-safe ML pipelines.

## Why this step is foundational

Most ML failures are not algorithm failures. They are definition failures: wrong target, leaked features, or train/test contamination.

If feature/target definitions are wrong, every metric that follows is misleading.

## Target variable requirements

A valid target must satisfy all three conditions:
- Clearly defined and measurable
- Present in historical labeled data
- Directly tied to a business decision or outcome

Examples:
- `Churn` -> binary classification
- `SalePrice` -> regression
- `DiagnosisCode` -> multi-class classification

In [4]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Ensure imports like `from src...` work when notebook runs from notebooks/
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TARGET_COLUMN
from src.data_loader import load_data

df = load_data(n_samples=500, random_state=42)
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
print("Target column from config:", TARGET_COLUMN)

Dataset shape: (500, 7)
Columns: ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'target']
Target column from config: target


## Golden rule for feature validity

A feature is valid only if it exists at the exact moment you make a real-world prediction.

If a column is created after the outcome occurs, it is leakage and must be excluded.

In [5]:
# Example feature config for this project dataset
NUMERICAL_FEATURES = [
    "feature_1",
    "feature_2",
    "feature_3",
    "feature_4",
    "feature_5",
    "feature_6",
]

CATEGORICAL_FEATURES = []
EXCLUDED_COLUMNS = []
ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

print("Feature count:", len(ALL_FEATURES))
print("Excluded columns:", EXCLUDED_COLUMNS)

Feature count: 6
Excluded columns: []


## Explicit X/y separation

Always separate features and target before fitting any preprocessing transform.

In [6]:
# Validation before separation
assert TARGET_COLUMN in df.columns, f"Target '{TARGET_COLUMN}' not found in dataset"
assert TARGET_COLUMN not in ALL_FEATURES, "Target leaked into feature list"
assert set(EXCLUDED_COLUMNS).isdisjoint(set(ALL_FEATURES)), "Excluded columns found in features"

missing_features = [c for c in ALL_FEATURES if c not in df.columns]
assert not missing_features, f"Missing features in data: {missing_features}"

X = df[ALL_FEATURES].copy()
y = df[TARGET_COLUMN].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target distribution:")
print(y.value_counts(normalize=True).rename("proportion"))

X shape: (500, 6)
y shape: (500,)
Target distribution:
target
1    0.502
0    0.498
Name: proportion, dtype: float64


## Leakage checks (practical)

This helper catches common early warnings:
- target accidentally included in features
- ID-like columns in features
- suspicious near-perfect correlation with target for numeric features

In [7]:
def validate_feature_definitions(dataframe: pd.DataFrame, target_col: str, feature_cols: list[str]) -> None:
    if target_col in feature_cols:
        raise ValueError(f"Target '{target_col}' is present in feature columns")

    missing = sorted(set(feature_cols) - set(dataframe.columns))
    if missing:
        raise ValueError(f"Features not found in dataframe: {missing}")

    id_like = [c for c in feature_cols if any(token in c.lower() for token in ["id", "key"]) ]
    if id_like:
        print("Warning: possible identifier columns found:", id_like)

    if pd.api.types.is_numeric_dtype(dataframe[target_col]):
        target_numeric = dataframe[target_col]
    else:
        target_numeric = pd.factorize(dataframe[target_col])[0]

    for col in feature_cols:
        if pd.api.types.is_numeric_dtype(dataframe[col]):
            corr = np.corrcoef(dataframe[col], target_numeric)[0, 1]
            if np.isfinite(corr) and abs(corr) > 0.95:
                print(f"Warning: '{col}' has very high correlation with target ({corr:.3f})")

    print("Feature definition validation completed successfully.")

validate_feature_definitions(df, TARGET_COLUMN, ALL_FEATURES)

Feature definition validation completed successfully.


## Correct sequence to prevent train/test contamination

Always follow this order:
1. Load data
2. Define target + eligible features
3. Separate `X` and `y`
4. Split train/test
5. Fit preprocessing on train only
6. Transform test with train-fitted preprocessors
7. Train model
8. Evaluate on test

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train/Test split complete without leakage.")
print("X_train:", X_train_scaled.shape, "X_test:", X_test_scaled.shape)

Train/Test split complete without leakage.
X_train: (400, 6) X_test: (100, 6)


## Project checklist for PR submission

Use this checklist before submitting:
- Target variable name, type, and business meaning documented
- Numerical and categorical features explicitly listed in config
- Excluded columns listed with reasons
- Assertion confirms target is not in features
- `X` and `y` separation shown in code
- Prediction-time availability validated for every feature
- Metrics aligned with problem type
- No hardcoded absolute paths

## Reflection

A model can only be as honest as the data definition behind it.

Define target and features first. Prevent leakage by design. Then train.